# NOTEBOOK 05: Neural Network & Hybrid Model

## MLP on scaled features; Hybrid = MLP(concat(scaled features, baseline class probabilities))

Loads scaled data and baseline model from Notebook 04. Trains MLP and Hybrid, evaluates on test, saves models.

In [1]:
import os
import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

In [2]:
CONFIG = {
    'data_dir': '../processed_data',
    'models_dir': '../models',
    'random_state': RANDOM_STATE,
    'device': torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    'batch_size': 256,
    'epochs': 200,
    'patience': 20,
    'lr': 1e-3,
    'weight_decay': 1e-4,
    'hidden_mlp': [256, 128, 64],
    'hidden_hybrid': [256, 128, 64],
    'dropout': 0.35,
    'num_classes': 3,
    'class_names': ['Poor', 'Moderate', 'Good'],
}

In [3]:
base = CONFIG['data_dir']
X_train_s = pd.read_csv(f"{base}/X_train_scaled.csv").values.astype(np.float32)
X_val_s = pd.read_csv(f"{base}/X_val_scaled.csv").values.astype(np.float32)
X_test_s = pd.read_csv(f"{base}/X_test_scaled.csv").values.astype(np.float32)
y_train = pd.read_csv(f"{base}/y_train.csv")['label'].values
y_val = pd.read_csv(f"{base}/y_val.csv")['label'].values
y_test = pd.read_csv(f"{base}/y_test.csv")['label'].values
n_features = X_train_s.shape[1]
baseline = joblib.load(f"{CONFIG['models_dir']}/best_baseline_model.joblib")
X_train_u = pd.read_csv(f"{base}/X_train.csv")
X_val_u = pd.read_csv(f"{base}/X_val.csv")
X_test_u = pd.read_csv(f"{base}/X_test.csv")
p_train = baseline.predict_proba(X_train_u)
p_val = baseline.predict_proba(X_val_u)
p_test = baseline.predict_proba(X_test_u)
X_train_hybrid = np.hstack([X_train_s, p_train]).astype(np.float32)
X_val_hybrid = np.hstack([X_val_s, p_val]).astype(np.float32)
X_test_hybrid = np.hstack([X_test_s, p_test]).astype(np.float32)
print(f"Features: {n_features}, Hybrid dim: {X_train_hybrid.shape[1]}")

Features: 34, Hybrid dim: 37


In [4]:
class MLP(nn.Module):
    def __init__(self, in_dim, num_classes, hidden, dropout):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(inplace=True), nn.Dropout(dropout)]
            prev = h
        self.backbone = nn.Sequential(*layers)
        self.head = nn.Linear(prev, num_classes)

    def forward(self, x):
        return self.head(self.backbone(x))

def train_loop(model, loader_val, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    for Xb, yb in loader_val:
        Xb, yb = Xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(Xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * Xb.size(0)
    return total_loss / len(loader_val.dataset)

def eval_loop(model, X, y, device):
    model.eval()
    with torch.no_grad():
        t = torch.from_numpy(X).float().to(device)
        logits = model(t)
        pred = logits.argmax(dim=1).cpu().numpy()
    return accuracy_score(y, pred), f1_score(y, pred, average='macro'), pred

In [5]:
def run_training(X_tr, y_tr, X_va, y_va, hidden, name):
    in_dim = X_tr.shape[1]
    model = MLP(in_dim, CONFIG['num_classes'], hidden, CONFIG['dropout']).to(CONFIG['device'])
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
    ds = TensorDataset(torch.from_numpy(X_tr).float(), torch.from_numpy(y_tr).long())
    loader = DataLoader(ds, batch_size=CONFIG['batch_size'], shuffle=True)
    best_val_acc = 0.0
    best_state = None
    wait = 0
    for ep in range(CONFIG['epochs']):
        train_loop(model, loader, criterion, optimizer, CONFIG['device'])
        va_acc, va_f1, _ = eval_loop(model, X_va, y_va, CONFIG['device'])
        if va_acc > best_val_acc:
            best_val_acc = va_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
        if wait >= CONFIG['patience']:
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    model = model.to(CONFIG['device'])
    return model

mlp_model = run_training(X_train_s, y_train, X_val_s, y_val, CONFIG['hidden_mlp'], 'MLP')
hybrid_model = run_training(X_train_hybrid, y_train, X_val_hybrid, y_val, CONFIG['hidden_hybrid'], 'Hybrid')

In [6]:
def full_eval(model, X, y, name):
    acc, f1, pred = eval_loop(model, X, y, CONFIG['device'])
    print(f"{name} — Accuracy: {acc:.4f}, Macro F1: {f1:.4f}")
    print(classification_report(y, pred, target_names=CONFIG['class_names']))
    print(confusion_matrix(y, pred))
    return acc, f1

b_pred = baseline.predict(X_test_u)
print("Test set — Baseline:", accuracy_score(y_test, b_pred), f1_score(y_test, b_pred, average='macro'))
print()
print("Test set — MLP & Hybrid:")
full_eval(mlp_model, X_test_s, y_test, 'MLP')
print()
full_eval(hybrid_model, X_test_hybrid, y_test, 'Hybrid')

Test set — Baseline: 0.6352365985681858 0.6354502133076495

Test set — MLP & Hybrid:
MLP — Accuracy: 0.6235, Macro F1: 0.6243
              precision    recall  f1-score   support

        Poor       0.67      0.73      0.70      3780
    Moderate       0.51      0.51      0.51      3894
        Good       0.70      0.63      0.66      3780

    accuracy                           0.62     11454
   macro avg       0.63      0.62      0.62     11454
weighted avg       0.62      0.62      0.62     11454

[[2766  836  178]
 [1051 1991  852]
 [ 295 1101 2384]]

Hybrid — Accuracy: 0.6311, Macro F1: 0.6334
              precision    recall  f1-score   support

        Poor       0.71      0.66      0.68      3780
    Moderate       0.52      0.55      0.53      3894
        Good       0.68      0.68      0.68      3780

    accuracy                           0.63     11454
   macro avg       0.64      0.63      0.63     11454
weighted avg       0.63      0.63      0.63     11454

[[2510 1017 

(0.6311332285664397, 0.6334333374883246)

In [7]:
os.makedirs(CONFIG['models_dir'], exist_ok=True)
torch.save(mlp_model.state_dict(), f"{CONFIG['models_dir']}/mlp_model.pt")
torch.save(hybrid_model.state_dict(), f"{CONFIG['models_dir']}/hybrid_model.pt")
meta = {'n_features': n_features, 'n_hybrid': X_train_hybrid.shape[1], 'hidden_mlp': CONFIG['hidden_mlp'], 'hidden_hybrid': CONFIG['hidden_hybrid'], 'num_classes': CONFIG['num_classes'], 'dropout': CONFIG['dropout']}
joblib.dump(meta, f"{CONFIG['models_dir']}/nn_meta.joblib")
print("Saved mlp_model.pt, hybrid_model.pt, nn_meta.joblib")

Saved mlp_model.pt, hybrid_model.pt, nn_meta.joblib
